In [1]:
%pip install pandas sqlalchemy ipython-sql jupysql matplotlib
%pip install "prettytable>=3.12.0"

  Using cached prettytable-3.17.0-py3-none-any.whl.metadata (34 kB)
Using cached prettytable-3.17.0-py3-none-any.whl (34 kB)
  Attempting uninstall: prettytable
    Found existing installation: prettytable 2.5.0
    Uninstalling prettytable-2.5.0:
      Successfully uninstalled prettytable-2.5.0
Note: you may need to restart the kernel to use updated packages.
  Using cached prettytable-2.5.0-py3-none-any.whl.metadata (22 kB)
Using cached prettytable-2.5.0-py3-none-any.whl (24 kB)
  Attempting uninstall: prettytable
    Found existing installation: prettytable 3.17.0
    Uninstalling prettytable-3.17.0:
      Successfully uninstalled prettytable-3.17.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupysql 0.11.1 requires prettytable>=3.12.0, but you have prettytable 2.5.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.

Note: you may need to restart the kernel to use updated packages.
ERROR: Exception:
Traceback (most recent call last):
  File "/Users/johnlagac/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/base_command.py", line 106, in _run_wrapper
    status = _inner_run()
  File "/Users/johnlagac/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/base_command.py", line 97, in _inner_run
    return self.run(options, args)
  File "/Users/johnlagac/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "/Users/johnlagac/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/commands/install.py", line 484, in run
    installed_versions[distribution.canonical_name] = distribution.version
  File "/Users/johnlagac/opt/anaconda3/lib/python3.8/site-packages/pip/_internal/metadata/pkg_resources.py", line 192, in version
    return parse_version(self._dist.version)
  File "/Users/johnlagac/opt/anacond

In [2]:
import sys
print(sys.version)
print(sys.executable)

3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 08:06:15) [Clang 14.0.6 ]
/Users/johnlagac/opt/anaconda3/envs/jupyter-clean/bin/python


# Advanced Demo: Five-Year Inventory Optimization and Pricing Strategy

This notebook demonstrates the Advanced goal of the Sari-Sari Store Simulator.

The Advanced goal is to:

- Generate five years of synthetic transaction data
- Add grocery-wide sales events that cause visible demand peaks
- Create monthly inventory snapshots and restock recommendations
- Generate monthly customer feedback
- Run an inventory optimization engine
- Run a pricing strategy engine
- Build a five-year dashboard with event peak detection

## 1. Set up project paths

Because this notebook is inside the `notebooks/` folder, we need to point Python back to the project root so it can import modules from `src/`.

In [3]:
from pathlib import Path
import sys

# The notebook is inside LT6_Final_Project/notebooks/
# Therefore, the project root is one folder above the current notebook folder.
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python path so src.* imports work.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/johnlagac/AIM_Code/DSC_512/LT6_Final_Project-main


In [4]:
# Define important project paths

INVENTORY_PATH   = PROJECT_ROOT / "data/raw/inventory.csv"
OUTPUT_FOLDER    = PROJECT_ROOT / "data/processed/advanced"
DATABASE_PATH    = PROJECT_ROOT / "src/database/sari_sari_store.db"

print("Inventory path :", INVENTORY_PATH)
print("Output folder  :", OUTPUT_FOLDER)
print("Database path  :", DATABASE_PATH)

Inventory path : /Users/johnlagac/AIM_Code/DSC_512/LT6_Final_Project-main/data/raw/inventory.csv
Output folder  : /Users/johnlagac/AIM_Code/DSC_512/LT6_Final_Project-main/data/processed/advanced
Database path  : /Users/johnlagac/AIM_Code/DSC_512/LT6_Final_Project-main/src/database/sari_sari_store.db


In [5]:
# Check that required input file exists

print("inventory.csv exists:", INVENTORY_PATH.exists())

if not INVENTORY_PATH.exists():
    raise FileNotFoundError(f"Missing file: {INVENTORY_PATH}")

inventory.csv exists: True


## 2. Preview the inventory

The Advanced level uses the same `data/raw/inventory.csv` as the master product list.
It is never modified — all outputs go to `data/processed/advanced/`.

In [6]:
import pandas as pd

inventory_raw = pd.read_csv(INVENTORY_PATH)
print(f"Products: {len(inventory_raw)}")
print(f"Categories: {sorted(inventory_raw['category'].unique())}")
display(inventory_raw)

Products: 15
Categories: ['Beverage', 'Food', 'Household', 'Personal Care', 'Snack']


,product_id,product_name,category,starting_stock,unit_cost,unit_price
0,P001,Coke 1L,Beverage,24,60,75
1,P002,Royal 1L,Beverage,18,58,72
2,P003,Bottled Water 500ml,Beverage,30,10,15
3,P004,Piattos Snack,Snack,36,14,18
4,P005,Nova Snack,Snack,32,13,18
5,P006,Lucky Me Beef Noodles,Food,40,10,14
6,P007,Pancit Canton Chilimansi,Food,40,12,16
7,P008,Canned Sardines,Food,25,22,28
8,P009,Corned Beef Small,Food,20,32,42
9,P010,Egg,Food,60,7,10


In [7]:
# Quick margin check — used by the pricing strategy engine
inventory_raw['unit_cost']  = pd.to_numeric(inventory_raw['unit_cost'],  errors='coerce')
inventory_raw['unit_price'] = pd.to_numeric(inventory_raw['unit_price'], errors='coerce')

inventory_raw['gross_margin'] = (
    (inventory_raw['unit_price'] - inventory_raw['unit_cost']) / inventory_raw['unit_price']
).round(3)

display(inventory_raw[['product_id','product_name','category','unit_cost','unit_price','gross_margin']]
        .sort_values('gross_margin', ascending=False))

,product_id,product_name,category,unit_cost,unit_price,gross_margin
12,P013,Shampoo Sachet,Personal Care,4,7,0.429
14,P015,Dishwashing Liquid Sachet,Household,6,10,0.400
11,P012,Coffee Sachet,Beverage,5,8,0.375
13,P014,Laundry Detergent Sachet,Household,5,8,0.375
2,P003,Bottled Water 500ml,Beverage,10,15,0.333
9,P010,Egg,Food,7,10,0.300
5,P006,Lucky Me Beef Noodles,Food,10,14,0.286
4,P005,Nova Snack,Snack,13,18,0.278
6,P007,Pancit Canton Chilimansi,Food,12,16,0.250
8,P009,Corned Beef Small,Food,32,42,0.238


## 3. Run the full Advanced pipeline

This runs all seven steps in order:

1. Generate five years of sales events
2. Generate 60 months of transactions (event-boosted)
3. Generate monthly outputs (transaction details, product summary, ledger summary, inventory before sales)
4. Generate customer feedback
5. Save monthly sales_events.csv per folder
6. Run the inventory optimizer
7. Run the pricing strategy engine and dashboard

Outputs are saved to `data/processed/advanced/` and `src/database/sari_sari_store.db`.

In [8]:
from src.advanced.advanced_runner import run_advanced_pipeline

results = run_advanced_pipeline(
    inventory_csv_path=INVENTORY_PATH,
    output_base_folder=OUTPUT_FOLDER,
    sqlite_db_path=DATABASE_PATH,
    random_seed=42,
    save_csv=True,
    save_sqlite=True,
    verbose=True,
)


ADVANCED GOAL: FULL FIVE-YEAR PIPELINE
Inventory source  : /Users/johnlagac/AIM_Code/DSC_512/LT6_Final_Project-main/data/raw/inventory.csv
Output folder     : /Users/johnlagac/AIM_Code/DSC_512/LT6_Final_Project-main/data/processed/advanced
SQLite database   : /Users/johnlagac/AIM_Code/DSC_512/LT6_Final_Project-main/src/database/sari_sari_store.db
Random seed       : 42

[Step 1 / 7] Generating sales events...

ADVANCED GOAL: SALES EVENT GENERATOR
Total events generated : 110
Average discount rate  : 13.5%
Average demand boost   : 1.40x

Events by category:
  Beverage             : 35 events
  Food                 : 30 events
  Snack                : 20 events
  Household            : 15 events
  Personal Care        : 10 events

Aggregated CSV saved  : /Users/johnlagac/AIM_Code/DSC_512/LT6_Final_Project-main/data/processed/advanced/sales_events.csv
SQLite table saved    : advanced_sales_events → /Users/johnlagac/AIM_Code/DSC_512/LT6_Final_Project-main/src/database/sari_sari_store.db



KeyError: "['product_name', 'category', 'unit_cost', 'unit_price'] not in index"

In [ ]:
# Extract all DataFrames from the results dictionary

sales_events            = results["sales_events"]
all_transactions        = results["all_transactions"]
all_transaction_details = results["all_transaction_details"]
all_product_summaries   = results["all_product_summaries"]
all_ledger_summaries    = results["all_ledger_summaries"]
all_feedback            = results["all_feedback"]
all_restock             = results["all_restock"]
pricing_recommendations = results["pricing_recommendations"]
dashboard               = results["dashboard"]

print("Sales events            :", len(sales_events))
print("Transactions (5yr)      :", len(all_transactions))
print("Transaction details     :", len(all_transaction_details))
print("Product summaries       :", len(all_product_summaries))
print("Ledger summaries        :", len(all_ledger_summaries))
print("Customer feedback       :", len(all_feedback))
print("Restock recommendations :", len(all_restock))
print("Pricing recommendations :", len(pricing_recommendations))
print("Dashboard rows          :", len(dashboard))

## 4. Check monthly folder output

Each `year_YYYY/month_MM/` folder should contain all eight files.

In [ ]:
# Spot-check one monthly folder — June 2022
sample_folder = OUTPUT_FOLDER / "year_2022" / "month_06"

expected_files = [
    "transactions.csv",
    "transaction_details.csv",
    "product_summary.csv",
    "ledger_summary.csv",
    "restock_recommendations.csv",
    "customer_feedback.csv",
    "sales_events.csv",
    "inventory_before_monthly_sales.csv",
]

print(f"Checking folder: {sample_folder}")
print()
for fname in expected_files:
    path = sample_folder / fname
    status = "OK" if path.exists() else "MISSING"
    print(f"  [{status}] {fname}")

In [ ]:
# Preview each file from the sample folder

for fname in expected_files:
    path = sample_folder / fname
    if path.exists():
        df = pd.read_csv(path)
        print(f"\n--- {fname} ({len(df)} rows) ---")
        display(df.head(3))

## 5. Inspect sales events

Sales events cause visible demand spikes during their active dates.
Each event affects one product category for a set number of days.

In [ ]:
display(sales_events.head(10))
print(f"\nTotal events : {len(sales_events)}")
print(f"Years covered: {sorted(pd.to_datetime(sales_events['start_date']).dt.year.unique())}")
print(f"\nEvents by category:")
print(sales_events.groupby('affected_category').size().sort_values(ascending=False))

In [ ]:
# Which events were active in June 2022?
events_june = sales_events[
    (pd.to_datetime(sales_events['start_date']) <= pd.Timestamp('2022-06-30')) &
    (pd.to_datetime(sales_events['end_date'])   >= pd.Timestamp('2022-06-01'))
]
display(events_june[['event_id','event_name','start_date','end_date','discount_rate','affected_category','demand_multiplier']])

## 6. Inspect five-year transactions

Transactions are generated daily for all 15 products across 60 months.
Demand is higher in December, on weekends, and during event periods.

In [ ]:
print("Shape:", all_transactions.shape)
print("Columns:", list(all_transactions.columns))
display(all_transactions.head())

In [ ]:
import matplotlib.pyplot as plt

all_transactions['transaction_date'] = pd.to_datetime(all_transactions['transaction_date'])
all_transactions['year']  = all_transactions['transaction_date'].dt.year
all_transactions['month'] = all_transactions['transaction_date'].dt.month

monthly_volume = (
    all_transactions
    .groupby(['year','month'])['quantity_sold']
    .sum()
    .reset_index()
)
monthly_volume['period'] = monthly_volume['year'].astype(str) + '-' + monthly_volume['month'].apply(lambda m: f'{m:02d}')

plt.figure(figsize=(14, 4))
plt.plot(range(len(monthly_volume)), monthly_volume['quantity_sold'], marker='o', markersize=3)
plt.xticks(
    range(0, len(monthly_volume), 6),
    monthly_volume['period'].iloc[::6],
    rotation=45, ha='right'
)
plt.title('Monthly Units Sold — 2022 to 2026')
plt.xlabel('Month')
plt.ylabel('Units Sold')
plt.tight_layout()
plt.show()

In [ ]:
# Verify seasonality: December should outsell February every year
comparison = monthly_volume[monthly_volume['month'].isin([2, 12])].pivot(index='year', columns='month', values='quantity_sold')
comparison.columns = ['February', 'December']
comparison['Dec vs Feb'] = (comparison['December'] / comparison['February']).round(2)
print("Seasonality check (December vs February):")
print(comparison)

## 7. Inspect transaction details and ledger summaries

In [ ]:
print("Transaction details shape:", all_transaction_details.shape)
print("Columns:", list(all_transaction_details.columns))
display(all_transaction_details.head())

In [ ]:
print("Ledger summaries shape:", all_ledger_summaries.shape)
display(all_ledger_summaries.head(12))

In [ ]:
annual = (
    all_ledger_summaries
    .groupby('year', as_index=False)
    .agg(
        annual_revenue=('total_revenue','sum'),
        annual_expense=('total_expense','sum'),
        annual_profit =('gross_profit', 'sum'),
    )
)
annual['margin'] = (annual['annual_profit'] / annual['annual_revenue']).round(3)
print("Annual financial summary:")
display(annual)

plt.figure(figsize=(8,4))
plt.bar(annual['year'].astype(str), annual['annual_profit'])
plt.title('Annual Gross Profit 2022–2026')
plt.xlabel('Year')
plt.ylabel('Gross Profit (PHP)')
plt.tight_layout()
plt.show()

## 8. Inspect customer feedback

In [ ]:
print("Shape:", all_feedback.shape)
display(all_feedback.head(10))

In [ ]:
sentiment_counts = all_feedback.groupby('sentiment').size().reset_index(name='count')
sentiment_counts['pct'] = (sentiment_counts['count'] / len(all_feedback) * 100).round(1)
print("Sentiment distribution:")
display(sentiment_counts)

plt.figure(figsize=(6,4))
plt.bar(sentiment_counts['sentiment'], sentiment_counts['count'], color=['steelblue','orange','salmon'])
plt.title('Customer Feedback Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
avg_rating = (
    all_feedback
    .groupby('product_name')['rating']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
avg_rating['rating'] = avg_rating['rating'].round(2)
print("Average rating by product:")
display(avg_rating)

## 9. Inspect inventory optimizer results

The optimizer classifies each product-month as CRITICAL, HIGH, MEDIUM, or LOW risk
and recommends a restock quantity.

In [ ]:
print("Shape:", all_restock.shape)
print("Columns:", list(all_restock.columns))
display(all_restock.head(10))

In [ ]:
risk_counts = all_restock.groupby('stockout_risk').size().reset_index(name='count')
risk_counts['pct'] = (risk_counts['count'] / len(all_restock) * 100).round(1)
print("Stockout risk distribution across all 60 months:")
display(risk_counts)

In [ ]:
critical = all_restock[all_restock['stockout_risk'] == 'CRITICAL'][['product_name','category','year','month','current_stock','average_monthly_quantity_sold','recommended_restock_quantity']]
print(f"CRITICAL stockout entries: {len(critical)}")
display(critical.head(10))

## 10. Inspect pricing strategy recommendations

The pricing engine analyses five years of demand trends and gross margins
to recommend price adjustments per product.

In [ ]:
print("Shape:", pricing_recommendations.shape)
display(pricing_recommendations)

In [ ]:
pricing_recommendations['price_change'] = (
    pricing_recommendations['recommended_price'] - pricing_recommendations['current_unit_price']
).round(2)
pricing_recommendations['change_pct'] = (
    pricing_recommendations['price_change'] / pricing_recommendations['current_unit_price'] * 100
).round(1)

increases = (pricing_recommendations['price_change'] > 0).sum()
decreases = (pricing_recommendations['price_change'] < 0).sum()
holds     = (pricing_recommendations['price_change'] == 0).sum()

print(f"Price increases recommended : {increases}")
print(f"Price decreases recommended : {decreases}")
print(f"Prices held (no change)     : {holds}")
print()
display(pricing_recommendations[['product_name','category','current_unit_price','recommended_price','price_change','change_pct','demand_trend','gross_margin','pricing_reason']])

## 11. Inspect the advanced dashboard

The dashboard is a normalized table with `event_flag=True` on months where
a grocery-wide sales event was active — making promo peaks visible.

In [ ]:
print("Dashboard shape:", dashboard.shape)
print("Metric groups:")
print(dashboard['metric_group'].unique())
display(dashboard.head(10))

In [ ]:
kpis = dashboard[dashboard['metric_group'] == 'kpi_summary'][['metric_name','value']]
print("Five-year KPI summary:")
display(kpis)

In [ ]:
trend = dashboard[
    (dashboard['metric_group'] == 'monthly_revenue_trend') &
    (dashboard['metric_name']  == 'monthly_revenue')
].copy()

trend['value'] = pd.to_numeric(trend['value'])
event_months   = trend[trend['event_flag'] == True]
normal_months  = trend[trend['event_flag'] == False]

plt.figure(figsize=(14, 4))
plt.plot(range(len(trend)), trend['value'], color='steelblue', linewidth=1.2, label='Monthly Revenue')
plt.scatter(
    trend.index[trend['event_flag'] == True] - trend.index[0],
    event_months['value'],
    color='orange', zorder=5, s=40, label='Sales Event Active'
)
plt.xticks(
    range(0, len(trend), 6),
    trend['dimension'].iloc[::6].values,
    rotation=45, ha='right'
)
plt.title('Monthly Revenue 2022–2026 (orange = event month)')
plt.xlabel('Month')
plt.ylabel('Revenue (PHP)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
cat_rev = dashboard[
    (dashboard['metric_group'] == 'sales_by_category') &
    (dashboard['metric_name']  == 'category_revenue_5yr')
][['dimension','value','rank']].sort_values('rank')

cat_rev['value'] = pd.to_numeric(cat_rev['value'])
print("Five-year revenue by category:")
display(cat_rev)

plt.figure(figsize=(7,4))
plt.bar(cat_rev['dimension'], cat_rev['value'])
plt.title('Five-Year Revenue by Category')
plt.xlabel('Category')
plt.ylabel('Revenue (PHP)')
plt.tight_layout()
plt.show()

In [ ]:
annual_dash = dashboard[
    (dashboard['metric_group'] == 'annual_summary') &
    (dashboard['metric_name']  == 'annual_gross_profit')
][['dimension','value']].rename(columns={'dimension':'year','value':'gross_profit'})
annual_dash['gross_profit'] = pd.to_numeric(annual_dash['gross_profit'])
print("Year-by-year gross profit from dashboard:")
display(annual_dash)

## 12. Connect to SQLite using SQL Magic

Now we inspect all Advanced tables directly with SQL queries.

In [ ]:
%load_ext sql

In [ ]:
# Connect to the shared database
%sql sqlite:///../src/database/sari_sari_store.db

In [ ]:
%%sql

SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;

## 13. Inspect SQLite tables

In [ ]:
%%sql

SELECT *
FROM advanced_sales_events
LIMIT 10;

In [ ]:
%%sql

SELECT *
FROM advanced_transactions_5yr
LIMIT 10;

In [ ]:
%%sql

SELECT *
FROM advanced_transaction_details_5yr
LIMIT 10;

In [ ]:
%%sql

SELECT *
FROM advanced_monthly_product_summary
WHERE year = 2022 AND month = 6
ORDER BY total_quantity_sold DESC;

In [ ]:
%%sql

SELECT *
FROM advanced_monthly_ledger_summary
ORDER BY year, month;

In [ ]:
%%sql

SELECT *
FROM advanced_customer_feedback
LIMIT 10;

In [ ]:
%%sql

SELECT *
FROM advanced_inventory_recommendations
WHERE stockout_risk IN ('CRITICAL','HIGH')
ORDER BY year, month
LIMIT 20;

In [ ]:
%%sql

SELECT *
FROM advanced_pricing_recommendations
ORDER BY product_id;

In [ ]:
%%sql

SELECT *
FROM advanced_monthly_dashboard_data
WHERE metric_group = 'kpi_summary';

## 14. SQL query — event peak months

This query pulls the months where a sales event was active and shows
the revenue boost compared to the five-year monthly average.

In [ ]:
%%sql

SELECT
    dimension                        AS month,
    ROUND(CAST(value AS FLOAT), 2)   AS monthly_revenue,
    event_flag
FROM advanced_monthly_dashboard_data
WHERE metric_group = 'monthly_revenue_trend'
  AND metric_name  = 'monthly_revenue'
ORDER BY month;

## 15. SQL ledger verification

Recalculate five-year totals directly from `advanced_transaction_details_5yr`
and compare with the stored ledger summaries.

In [ ]:
%%sql

SELECT
    SUM(revenue)      AS total_revenue_calc,
    SUM(expense)      AS total_expense_calc,
    SUM(gross_profit) AS total_profit_calc
FROM advanced_transaction_details_5yr;

In [ ]:
%%sql

SELECT
    SUM(total_revenue) AS total_revenue_ledger,
    SUM(total_expense) AS total_expense_ledger,
    SUM(gross_profit)  AS total_profit_ledger
FROM advanced_monthly_ledger_summary;

## 16. Final Advanced goal validation

In [ ]:
checks = {
    "sales_events_generated":        len(sales_events) > 0,
    "transactions_generated":        len(all_transactions) > 0,
    "transaction_details_generated": len(all_transaction_details) > 0,
    "product_summaries_generated":   len(all_product_summaries) > 0,
    "ledger_summaries_generated":    len(all_ledger_summaries) > 0,
    "feedback_generated":            len(all_feedback) > 0,
    "restock_recommendations":       len(all_restock) > 0,
    "pricing_recommendations":       len(pricing_recommendations) > 0,
    "dashboard_generated":           len(dashboard) > 0,
    "60_months_covered":             all_ledger_summaries.shape[0] == 60,
    "event_peaks_flagged":           dashboard[dashboard['event_flag'] == True].shape[0] > 0,
    "monthly_folder_files_exist":    (OUTPUT_FOLDER / 'year_2022' / 'month_06' / 'transaction_details.csv').exists(),
    "inventory_before_exists":       (OUTPUT_FOLDER / 'year_2022' / 'month_06' / 'inventory_before_monthly_sales.csv').exists(),
    "sqlite_db_exists":              DATABASE_PATH.exists(),
}

print("Advanced goal validation checklist:")
print()
all_passed = True
for check, result in checks.items():
    status = 'PASS' if result else 'FAIL'
    if not result:
        all_passed = False
    print(f"  [{status}] {check}")

print()
if all_passed:
    print("Advanced goal completed successfully.")
else:
    print("Some checks failed — review the output above.")

## End of Advanced Demo

This notebook demonstrated the Advanced level of the Sari-Sari Store Simulator.

The main project logic is stored in `.py` files under `src/advanced/`.

This notebook is only used for demonstration, validation, SQL Magic inspection, and visualizations.